# Nonlinear Spatial Models

This guide demonstrates three nonlinear spatial models in `bayespecon` on synthetic data.

## Spatial Probit

For binary outcomes $y_i \in \{0,1\}$, latent utilities $y^*$ follow a spatial lag:

$$y^* = \rho W y^* + X\beta + a + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

where $a$ are region-level random effects and $y_i = \mathbf{1}[y^*_i > 0]$.

## SAR Tobit

For censored outcomes, the spatial autoregressive Tobit specifies:

$$y^* = \rho W y^* + X\beta + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$
$$y_i = \max(c,\, y^*_i)$$

where $c$ is the censoring threshold (default 0).

## SEM Tobit

The spatial error Tobit places spatial dependence in the disturbance:

$$y^* = X\beta + u, \quad u = \lambda W u + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$
$$y_i = \max(c,\, y^*_i)$$

In [ ]:
import numpy as np
from scipy.special import erf
from libpysal.graph import Graph
from bayespecon import SpatialProbit, SARTobit, SEMTobit

rng = np.random.default_rng(1234)

def make_line_W(n):
    W = np.zeros((n, n))
    for i in range(n):
        if i > 0:
            W[i, i - 1] = 1.0
        if i < n - 1:
            W[i, i + 1] = 1.0
    s = W.sum(axis=1, keepdims=True)
    return W / np.where(s == 0, 1, s)

def W_to_graph(W):
    i, j = np.where(W != 0)
    return Graph.from_arrays(i, j, W[i, j].astype(float))

In [ ]:
# SpatialProbit
m, n_per_region = 8, 20
Wm = make_line_W(m)
Gm = W_to_graph(Wm)
rho, beta, sigma_a = 0.35, np.array([0.3, 1.0]), 0.8
a = np.linalg.solve(np.eye(m) - rho * Wm, sigma_a * rng.standard_normal(m))
region_ids = np.repeat(np.arange(m), n_per_region)
x1 = rng.standard_normal(m * n_per_region)
X = np.column_stack([np.ones_like(x1), x1])
eta = X @ beta + a[region_ids]
p = 0.5 * (1.0 + erf(eta / np.sqrt(2.0)))
y = rng.binomial(1, p).astype(float)
sp = SpatialProbit(y=y, X=X, W=Gm, region_ids=region_ids)
sp.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)
sp.summary(var_names=["rho", "beta", "sigma_a"])

In [ ]:
# SARTobit and SEMTobit
n = 30
W = make_line_W(n)
G = W_to_graph(W)
X = np.column_stack([np.ones(n), rng.standard_normal(n)])
beta = np.array([1.0, 1.5])
sigma, rho, lam = 0.8, 0.4, 0.35
y_lat_sar = np.linalg.solve(np.eye(n) - rho * W, X @ beta + sigma * rng.standard_normal(n))
y_sar = np.maximum(0.0, y_lat_sar)
sar_tobit = SARTobit(y=y_sar, X=X, W=G, censoring=0.0)
sar_tobit.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)

u_sem = np.linalg.solve(np.eye(n) - lam * W, sigma * rng.standard_normal(n))
y_lat_sem = X @ beta + u_sem
y_sem = np.maximum(0.0, y_lat_sem)
sem_tobit = SEMTobit(y=y_sem, X=X, W=G, censoring=0.0)
sem_tobit.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)

sar_tobit.summary(var_names=["rho", "beta", "sigma"]), sem_tobit.summary(var_names=["lam", "beta", "sigma"])